# Классификация глазных заболеваний
**Курс:** Основы теории принятия решений  
**Датасет:** gunavenkatdoddi/eye-diseases-classification (Kaggle)  
**4 класса:** Normal, Cataract, Glaucoma, Diabetic Retinopathy  

### Порядок запуска
Выполняй ячейки **строго по порядку** сверху вниз.

In [ ]:
# ── Ячейка 1: Установка зависимостей и клонирование репо ──────────────────
# Первый запуск занимает ~1 минуту
!pip install torch torchvision scikit-learn matplotlib seaborn kaggle tqdm -q

# Клонируй свой репозиторий или загрузи файлы вручную через Files → Upload
# !git clone https://github.com/ВАШ_USERNAME/eye-classification.git
# %cd eye-classification

import sys, os
sys.path.insert(0, ".")
print("Python:", sys.version[:6])

In [ ]:
# ── Ячейка 2: Настройка Kaggle API ────────────────────────────────────────
# Вариант A: загрузи kaggle.json через Files → Upload, затем выполни эту ячейку
import os
os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json 2>/dev/null || echo "kaggle.json не найден — используй Вариант Б"
!chmod 600 /root/.kaggle/kaggle.json 2>/dev/null || true

# Вариант Б: вставь credentials напрямую
# with open("/root/.kaggle/kaggle.json", "w") as f:
#     f.write('{"username":"ВАШ_USERNAME","key":"ВАШ_API_KEY"}')
# !chmod 600 /root/.kaggle/kaggle.json

print("Kaggle API настроен")

In [ ]:
# ── Ячейка 3: Монтирование Google Drive (для сохранения результатов) ───────
from google.colab import drive
drive.mount("/content/drive")
print("Google Drive смонтирован")

In [ ]:
# ── Ячейка 4: Импорты и конфигурация ──────────────────────────────────────
import torch
from config import ExperimentConfig
from src.data_loader import EyeDataset
from src.experiment import MonteCarloRunner
from src.evaluate import Evaluator
from src.cnn_model import EyeCNN
from src.trainer import Trainer

cfg = ExperimentConfig()

print(f"Device:       {cfg.device}")
print(f"Image sizes:  {cfg.img_sizes}")
print(f"SNR levels:   {cfg.snr_levels_db}")
print(f"Repetitions:  {cfg.n_repetitions}")
print(f"Epochs:       {cfg.num_epochs}")
if cfg.device == "cpu":
    print("⚠️  GPU не найден. Смени Runtime: Runtime → Change runtime type → GPU")

In [ ]:
# ── Ячейка 5: Загрузка и проверка данных ──────────────────────────────────
dataset = EyeDataset(cfg)
dataset.download()  # пропускает если данные уже есть

# Проверяем структуру на разрешении 128×128
train_loader, test_loader = dataset.get_loaders(img_size=128)

images, labels = next(iter(train_loader))
print(f"Классы:           {dataset.class_names}")
print(f"Батч изображений: {images.shape}")
print(f"Обучающих батчей: {len(train_loader)}")
print(f"Тестовых батчей:  {len(test_loader)}")

In [ ]:
# ── Ячейка 6: Серия 1 — Монте-Карло: точность vs SNR ─────────────────────
# ⏱ Время: ~1.5 часа на GPU Colab (15 повторений, 2 размера, 20 эпох каждая)
# Для быстрой проверки: cfg.n_repetitions = 2; cfg.num_epochs = 1

runner = MonteCarloRunner(cfg)
results_by_size: dict[int, dict] = {}

for img_size in cfg.img_sizes:
    print(f"\n{'='*50}")
    print(f"Размер изображения: {img_size}×{img_size}")
    print(f"{'='*50}")
    results_by_size[img_size] = runner.run(img_size=img_size, train_fraction=1.0)

print("\nСерия 1 завершена!")

In [ ]:
# ── Ячейка 7: Серия 2 — точность vs число обучающих выборок ──────────────
# Фиксированный SNR = 10 дБ, размер 128×128
# ⏱ Время: ~45 минут (3 фракции × 15 повторений)

print(f"SNR фиксирован: {cfg.fixed_snr_for_samples_exp} дБ")
print(f"Фракции:        {cfg.train_fractions}")

results_by_fraction = runner.run_samples_experiment(img_size=128)
print("\nСерия 2 завершена!")

In [ ]:
# ── Ячейка 8: Построение графиков ─────────────────────────────────────────
evaluator = Evaluator(cfg, class_names=dataset.class_names)

# График 1: Точность vs SNR для обоих классификаторов и обоих размеров
print("График 1: Точность vs SNR")
evaluator.accuracy_vs_snr(results_by_size)

# График 2: Точность vs число обучающих выборок
print("График 2: Точность vs число выборок")
evaluator.accuracy_vs_samples(results_by_fraction, img_size=128)

In [ ]:
# ── Ячейка 9: Матрицы ошибок ──────────────────────────────────────────────
# Обучаем финальные модели на 100% данных (128×128)
trainer = Trainer(cfg)
train_loader_128, test_loader_128 = dataset.get_loaders(img_size=128)
centroids = dataset.compute_centroids(train_loader_128)

print("Обучаем финальную CNN...")
final_cnn = trainer.train_cnn(EyeCNN(128, num_classes=cfg.num_classes), train_loader_128)

print("Подгоняем оптимальный классификатор...")
final_optimal = trainer.fit_optimal(centroids)

# Матрицы при двух уровнях шума
for snr_db in [20.0, 0.0]:
    noise_label = "мало шума" if snr_db == 20.0 else "много шума"
    evaluator.plot_confusion_matrix(
        final_cnn, test_loader_128, snr_db,
        title=f"CNN SNR={snr_db}dB {noise_label}",
    )
    evaluator.plot_confusion_matrix(
        final_optimal, test_loader_128, snr_db,
        title=f"Optimal SNR={snr_db}dB {noise_label}",
    )

In [ ]:
# ── Ячейка 10: Сохранение результатов на Google Drive ─────────────────────
evaluator.save_to_drive()

import os
from pathlib import Path
results_path = Path(cfg.results_dir)
all_files = list(results_path.rglob("*.png"))
print(f"\nВсего сохранённых файлов: {len(all_files)}")
for f in sorted(all_files):
    print(f"  {f}")